# Series Data Exploration

This example shows some processing of `PupilSeries`.

In [ ]:
import matplotlib.pyplot as plt
import piprkit

ps = piprkit.load_real_series()

## Load data

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 2))
ps.plot(ax=ax, label="Raw Pupil Trace", color="darkblue", alpha=0.7)
ps.plot_light_stimulus(ax=ax, color="yellow", alpha=0.3, label="_Light Stimuli")
ax.set_title("Step 1: Raw Series Data")
ax.set_ylabel("Pupil Size (mm)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Smoothing and limiting rate of change

The data is smoothened using a rolling mean of 0.1 seconds using `rolling_mean(0.1)`, followed by limiting the rate of change to a maximum of 6 mm/s (Amiot et al., 2024) using `limit_rate_of_change`.

In [ ]:
ps_rate_limited = ps.filters.rolling_mean(0.1, inplace=False)
ps_rate_limited.filters.limit_rate_of_change()
ps_rate_limited.drop_nan()

fig, ax = plt.subplots(1, 1, figsize=(8, 2))
ps_rate_limited.plot(ax=ax, label="Rate Limited (< 6 mm/s)", color="orange", alpha=0.8)
ps_rate_limited.plot_light_stimulus(ax=ax, color="yellow", alpha=0.3, label="_Light Stimuli")
ax.set_title("Step 2: Rate Limiting (Artifact Removal)")
ax.set_ylabel("Pupil Size (mm)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"Data points removed by rate limiting: {len(ps.get_time()) - len(ps_rate_limited.get_time())}")
print(f"Percentage of data retained: {100 * len(ps_rate_limited.get_time()) / len(ps.get_time()):.1f}%")

## Step 3: Smoothing

Using `rolling_mean(0.5)`, each sample is replaced by the mean of the samples 0.25 s before and after it.

In [ ]:
ps_smoothed = ps_rate_limited.filters.rolling_mean(0.5, inplace=False)

fig, ax = plt.subplots(1, 1, figsize=(8, 2))
ps_smoothed.plot(ax=ax, label="Smoothed (0.5s window)", color="purple", alpha=0.8)
ps_smoothed.plot_light_stimulus(ax=ax, color="yellow", alpha=0.3, label="_Light Stimuli")
ax.set_title("Step 3: Smoothing Filter")
ax.set_ylabel("Pupil Size (mm)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Step 4: Manual removal of the remaining blink

A blink still exists around 155 seconds. We can manually add a blink using `add_blinks`. Removal is again done in the same manner as before using `remove_blinks` and `interpolate`.

In [ ]:
# Manual removal of the last "surviving" blink.
# We can see a blink at the following spot:
blink_locations = [
    (153.5, 156.5),
]
ps_final = ps_smoothed.copy()
ps_final.add_blinks(blink_locations)
ps_final.filters.remove_blinks()
ps_final.drop_nan()
ps_final.interpolate(new_time=ps.get_time())

fig, ax = plt.subplots(1, 1, figsize=(8, 2))
ps_final.plot(ax=ax, label="Final (Post-Processing)", color="purple", alpha=0.8)
ps_final.plot_light_stimulus(
    ax=ax, color="yellow", alpha=0.3, label="_Light Stimuli"
)
ax.set_ylabel("Pupil Size (mm)")
ax.legend()
plt.show()